### Programming Assignment #3: Letter Recognition
In this programming assignment, we will load the letter-recognition.data.csv file, explore the data set, and then do the letter classifications. We will train multiple classifiers and apply the ensemble learning to improve classification results.

#### Data Set Information:
The objective is to identify each of a large number of black-and-white rectangular pixel displays as one of the 26 capital letters in the English alphabet. The character images were based on 20 different fonts and each letter within these 20 fonts was randomly distorted to produce a file of 20,000 unique stimuli. Each stimulus was converted into 16 primitive numerical attributes (statistical moments and edge counts) which were then scaled to fit into a range of integer values from 0 through 15. We train on the first 16000 items and then use the resulting model to predict the letter category for the remaining 4000.

#### Features:
1. letter capital letter (26 values from A to Z)
2. x-box horizontal position of box (integer)
3. y-box vertical position of box (integer)
4. width width of box (integer)
5. height height of box (integer)
6. onpix total # on pixels (integer)
7. x-bar mean x of on pixels in box (integer)
8. y-bar mean y of on pixels in box (integer)
9. x2bar mean x variance (integer)
10. y2bar mean y variance (integer)
11. xybar mean x y correlation (integer)
12. x2ybr mean of x*x*y (integer)
13. xy2br mean of x*y*y (integer)
14. x-ege mean edge count left to right (integer)
15. xegvy correlation of x-ege with y (integer)
16. y-ege mean edge count bottom to top (integer)
17. yegvx correlation of y-ege with x (integer)

### The grading rubric is in the following text and code blocks. Everywhere we need to fill in code, the number points is displayed.

### Note, before we start, we need to upload the data file letter-recognition.data.csv.


In [1]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import os

## Read data and data preprocessing.

In [2]:
import pandas as pd

def load_data(path, name):
    csv_path = os.path.join(path, name)
    return pd.read_csv(csv_path, header=None)

In [3]:
letters = load_data(".", "letter-recognition.data.csv")

In [4]:
letters.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


In [5]:
letters.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       20000 non-null  object
 1   1       20000 non-null  int64 
 2   2       20000 non-null  int64 
 3   3       20000 non-null  int64 
 4   4       20000 non-null  int64 
 5   5       20000 non-null  int64 
 6   6       20000 non-null  int64 
 7   7       20000 non-null  int64 
 8   8       20000 non-null  int64 
 9   9       20000 non-null  int64 
 10  10      20000 non-null  int64 
 11  11      20000 non-null  int64 
 12  12      20000 non-null  int64 
 13  13      20000 non-null  int64 
 14  14      20000 non-null  int64 
 15  15      20000 non-null  int64 
 16  16      20000 non-null  int64 
dtypes: int64(16), object(1)
memory usage: 2.6+ MB


In [6]:
col_names = ['letter', 'x-box', 'y-box', 'width', 'height', 'onpix', 'x-bar', 'y-bar',
             'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br', 'x-ege', 'xegvy', 'y-ege', 'yegvx']

In [7]:
print(col_names)
print(len(col_names))

['letter', 'x-box', 'y-box', 'width', 'height', 'onpix', 'x-bar', 'y-bar', 'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br', 'x-ege', 'xegvy', 'y-ege', 'yegvx']
17


In [8]:
letters.columns = col_names

In [9]:
letters.head()

,letter,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


In [10]:
letters['letter']

0        T
1        I
2        D
3        N
4        G
        ..
19995    D
19996    C
19997    T
19998    S
19999    A
Name: letter, Length: 20000, dtype: object

Convert letters better A and Z to numbers between 0 to 25.

In [11]:
a = np.zeros(len(letters))
for i in range(len(letters)):
    a[i] = ord(letters['letter'][i]) - ord('A')

In [12]:
a

array([19.,  8.,  3., ..., 19., 18.,  0.])

Add the numbered labels as a column named y.

In [13]:
letters['y'] = a

In [14]:
letters.head()

,letter,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx,y
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8,19.0
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10,8.0
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9,3.0
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8,13.0
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10,6.0


Drop column 'letter' since do not need it any more.

In [15]:
letters_new = letters.drop('letter', axis=1)

# Prepare training set (the first 16000 samples) and test set (the last 4000 samples).

In [16]:
letters.head()

,letter,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx,y
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8,19.0
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10,8.0
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9,3.0
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8,13.0
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10,6.0


In [17]:
letters_new.head()

,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx,y
0,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8,19.0
1,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10,8.0
2,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9,3.0
3,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8,13.0
4,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10,6.0


In [18]:
X_train = letters_new.iloc[0:16000,:16]
X_train.head()

,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
0,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


In [19]:
X_train.shape

(16000, 16)

In [20]:
y_train = letters_new.iloc[0:16000,16]
y_train.head()

0    19.0
1     8.0
2     3.0
3    13.0
4     6.0
Name: y, dtype: float64

In [21]:
y_train.shape

(16000,)

In [22]:
X_test = letters_new.iloc[16000:,:16]
X_test.head()

,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
16000,4,10,6,7,9,9,6,4,3,6,7,7,9,8,5,6
16001,6,9,8,4,3,8,7,3,4,13,5,8,6,8,0,8
16002,6,9,8,8,10,7,7,5,4,7,6,8,7,9,7,10
16003,5,6,6,4,3,7,6,2,7,7,6,9,0,9,4,8
16004,5,9,7,6,4,9,7,3,5,10,4,6,5,8,1,7


In [23]:
X_test.shape

(4000, 16)

In [24]:
y_test = letters_new.iloc[16000:,16]
y_test.head()

16000    20.0
16001    13.0
16002    21.0
16003     8.0
16004    13.0
Name: y, dtype: float64

In [25]:
y_test.shape

(4000,)

# Try three classifiers: (i) random forest, (ii) SVM, and (iii) MLP. Then use an ensemble method through soft-voting.

## Classifiers 1: random forest

### Scale the training data set.

In [26]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float64))

### Use default parameters in random forest to get a sense of the results.

Use 100 estimators and set random state to 42 to obtain a random forest classifier. (10 points)

In [28]:
from sklearn.ensemble import RandomForestClassifier

# fill in code

rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)

Display the corss validation result. Use 3-fold validation. (10 points)

In [29]:
from sklearn.model_selection import cross_val_score
# fill in code
cross_val_score(rnd_clf, X_train_scaled, y_train, cv=3)

array([0.95444319, 0.95087193, 0.95349709])

## Classifiers 2: MLP

Obtain a MLPClassifier using random state 42. (10 points)

In [35]:
from sklearn.neural_network import MLPClassifier
# fill in code
mlp_clf = MLPClassifier(random_state=42)

Display the cross validation result using 3-fold cross validation. (5 points)

In [36]:
# fill in code below
cross_val_score(mlp_clf, X_train_scaled, y_train, cv=3)

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.94131984, 0.94130883, 0.93905869])

### Classifiers 3: SVM (one vs all)

Obtain a svm classifier with random state 42.

In [32]:
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier

ovr_clf = OneVsRestClassifier(SVC(gamma="auto", random_state=42, probability=True))

Display cross validation result with 3-fold cross validation. (5 points).

In [33]:
# fill in code below
cross_val_score(ovr_clf, X_train_scaled, y_train, cv=3)

array([0.9135733 , 0.91449466, 0.91318207])

#Hard-voting

Construct a hard-voting classifer using the above three classifiers. (10 points)

In [40]:
from sklearn.ensemble import VotingClassifier
# complete the code below
voting_clf_hard = VotingClassifier(
    estimators=[('rf', rnd_clf), ('mlp', mlp_clf), ('ovr', ovr_clf)],
    voting='hard')

Display the cross validation result of the hard-voting classifier using 3-fold cross validation. (5 points)

In [41]:
# fill in code below
cross_val_score(voting_clf_hard, X_train_scaled, y_train, cv=3)

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.95256843, 0.95012188, 0.94918432])

# Soft-voting

Construct a soft-voting classifier using the above three classifiers and train the classifier. (10 points)

In [42]:
from sklearn.ensemble import VotingClassifier
# complete the code below
voting_clf_soft = VotingClassifier(
    estimators=[('rf', rnd_clf), ('mlp', mlp_clf), ('ovr', ovr_clf)],
    voting='soft')

voting_clf_soft.fit(X_train_scaled, y_train)

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


VotingClassifier(estimators=[('rf', RandomForestClassifier(random_state=42)),
                             ('mlp', MLPClassifier(random_state=42)),
                             ('ovr',
                              OneVsRestClassifier(estimator=SVC(gamma='auto',
                                                                probability=True,
                                                                random_state=42)))],
                 voting='soft')

Display the cross validation result of the soft-voting classifier using 3-fold cross validation. (5 points)

In [43]:
# fill in code below
cross_val_score(voting_clf_soft, X_train_scaled, y_train, cv=3)

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.95650544, 0.95330958, 0.95012188])

Check the accuracy of the soft-voting classifier using test data. (10 points)

In [46]:
# fill in code below
from sklearn.metrics import accuracy_score

X_test_scaled = scaler.transform(X_test)
print(voting_clf_soft.__class__.__name__, voting_clf_soft.score(X_test_scaled, y_test))

VotingClassifier 0.96425


# Stacking

We will use stacking to further improve the result.

First we use the StackingClassifier implemented by sklearn.

In [47]:
from sklearn.ensemble import StackingClassifier

stacking_clf = StackingClassifier(
    estimators=[
        ('mlp', MLPClassifier(random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svc', OneVsRestClassifier(SVC(gamma="auto", random_state=42, probability=True)))
    ],
    final_estimator=RandomForestClassifier(random_state=43),
    cv=5  # number of cross-validation folds
)
stacking_clf.fit(X_train_scaled, y_train)

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multi

StackingClassifier(cv=5,
                   estimators=[('mlp', MLPClassifier(random_state=42)),
                               ('rf', RandomForestClassifier(random_state=42)),
                               ('svc',
                                OneVsRestClassifier(estimator=SVC(gamma='auto',
                                                                  probability=True,
                                                                  random_state=42)))],
                   final_estimator=RandomForestClassifier(random_state=43))

Obtain the classification result on test data.

In [48]:
X_test_scaled = scaler.transform(X_test.astype(np.float64))
from sklearn.metrics import accuracy_score

y_test_pred_stacking = stacking_clf.predict(X_test_scaled)
print(stacking_clf.__class__.__name__, accuracy_score(y_test, y_test_pred_stacking))

StackingClassifier 0.972


The result is great! It beats both soft-voting and hard-voting.

# Next, we will try to implement stacking by ourselves and see whether we can beat sklearn.

Scale the data set X_train.

In [49]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float64))

Train a random forest classifier using (X_train, y_train).

In [50]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rnd_clf.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

Obtain predictions using cross_val_predict with random forest classifier.

In [51]:
from sklearn.model_selection import cross_val_predict

y_pred_rf = cross_val_predict(
    rnd_clf,
    X_train_scaled,
    y_train,
    cv=5,
    method="predict"
)

Train a MLP classifier.

In [52]:
from sklearn.neural_network import MLPClassifier
mlp_clf = MLPClassifier(random_state=42)
mlp_clf.fit(X_train_scaled, y_train)

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(random_state=42)

Obtain predictions using cross_val_predict with MLP classifier.

In [53]:
from sklearn.model_selection import cross_val_predict

y_pred_mlp = cross_val_predict(
    mlp_clf,
    X_train_scaled,
    y_train,
    cv=5,
    method="predict"
)

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multi

Train a SVC classifier.

In [54]:
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier

ovr_clf = OneVsRestClassifier(SVC(gamma="auto", random_state=42, probability=True))
ovr_clf.fit(X_train_scaled, y_train)

OneVsRestClassifier(estimator=SVC(gamma='auto', probability=True,
                                  random_state=42))

Obtain predictions using cross_val_preidct with SVM classifier.

In [55]:
from sklearn.model_selection import cross_val_predict

y_pred_ovr = cross_val_predict(
    ovr_clf,
    X_train_scaled,
    y_train,
    cv=5,
    method="predict"
)

Combine predictions from 3 predictors.

In [56]:
X_stack_train=np.c_[y_pred_rf, y_pred_mlp, y_pred_ovr]
print(X_stack_train.shape)

(16000, 3)


In [57]:
print(X_stack_train)

[[19. 19. 19.]
 [ 8.  8.  8.]
 [ 3.  3.  3.]
 ...
 [ 6.  6.  6.]
 [ 4.  4.  4.]
 [ 2.  2.  2.]]


Try random forest as blender.

In [58]:
blending_rnd_clf = RandomForestClassifier(n_estimators=100, random_state=43)
blending_rnd_clf.fit(X_stack_train, y_train)

RandomForestClassifier(random_state=43)

Display the 3-fold cross validation result.

In [59]:
from sklearn.model_selection import cross_val_score
cross_val_score(blending_rnd_clf, X_stack_train, y_train, cv=3, scoring="accuracy")

array([0.95350581, 0.95180949, 0.94937184])

Try MLP as blender and display the 3-fold cross validation result.

In [60]:
blending_mlp_clf = MLPClassifier(random_state=43)
blending_mlp_clf.fit(X_stack_train, y_train)
cross_val_score(blending_mlp_clf, X_stack_train, y_train, cv=3, scoring="accuracy")

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.76227972, 0.86480405, 0.82617664])

We can see the MLP blender is not performing very well.

The reason is that we are treat predictions from the three predictors as numerical data instead of categorical data.

The above approach is incorrect. Instead, we should use onehot encoder to encode these predictions.

In [61]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder()
X_stack_train_1hot = onehot_encoder.fit_transform(X_stack_train)

In [62]:
X_stack_train_1hot.shape

(16000, 78)

In [63]:
blending_rnd_clf_1hot = RandomForestClassifier(n_estimators=100, random_state=43)
blending_rnd_clf_1hot.fit(X_stack_train_1hot, y_train)
cross_val_score(blending_rnd_clf_1hot, X_stack_train_1hot, y_train, cv=3, scoring="accuracy")

array([0.9583802 , 0.95574723, 0.95405963])

We see improvements for random forest blender.

We next try MLP blender.

In [64]:
blending_mlp_clf_1hot = MLPClassifier(random_state=43)
# fit the MLP blender
blending_mlp_clf_1hot.fit(X_stack_train_1hot, y_train)
# display cross validtion result
cross_val_score(blending_mlp_clf_1hot, X_stack_train_1hot, y_train, cv=3, scoring="accuracy")

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.95538058, 0.95387212, 0.95330958])

We can see the MLP blender is performing much better now. Both MLP blender and Random Forest blender are better than the hard voting.

However, it seems Random Forest blender still is a little better. We use random forest as the blender for the test data.

In [65]:
X_test_scaled = scaler.transform(X_test.astype(np.float64))
y_test_rnd = rnd_clf.predict(X_test_scaled)
y_test_mlp = mlp_clf.predict(X_test_scaled)
y_test_ovr = ovr_clf.predict(X_test_scaled)
X_stack_test = np.c_[y_test_rnd,y_test_mlp,y_test_ovr]
X_stack_test_1hot = onehot_encoder.transform(X_stack_test)
y_test_blending = blending_rnd_clf_1hot.predict(X_stack_test_1hot)
print(blending_rnd_clf_1hot.__class__.__name__, accuracy_score(y_test, y_test_blending))

RandomForestClassifier 0.96425


The stacking classifer is better than each of the individual classifiers. It is as good as the soft voting classifer, it is better than the hard voting classifier.

How do we improve further? Use predict_proba instead of predict.


In [66]:
rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rnd_clf.fit(X_train_scaled, y_train)
y_pred_rf = cross_val_predict(
    rnd_clf,
    X_train_scaled,
    y_train,
    cv=5,
    method="predict_proba"
)

In [67]:
mlp_clf = MLPClassifier(random_state=42)
mlp_clf.fit(X_train_scaled, y_train)
y_pred_mlp = cross_val_predict(
    mlp_clf,
    X_train_scaled,
    y_train,
    cv=5,
    method="predict_proba"
)

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multi

In [68]:
ovr_clf = OneVsRestClassifier(SVC(gamma="auto", random_state=42, probability=True))
ovr_clf.fit(X_train_scaled, y_train)
y_pred_ovr = cross_val_predict(
    ovr_clf,
    X_train_scaled,
    y_train,
    cv=5,
    method="predict_proba"
)

In [69]:
X_stack_train=np.c_[y_pred_rf, y_pred_mlp, y_pred_ovr]
print(X_stack_train.shape)

(16000, 78)


In [70]:
blending_rnd_clf = RandomForestClassifier(n_estimators=100, random_state=43)
blending_rnd_clf.fit(X_stack_train, y_train)

RandomForestClassifier(random_state=43)

In [71]:
from sklearn.model_selection import cross_val_score
cross_val_score(blending_rnd_clf, X_stack_train, y_train, cv=3, scoring="accuracy")

array([0.96775403, 0.96362273, 0.96193512])

Train the MLP blender and display 3-fold cross valdation result. (10 points)

In [72]:
blending_mlp_clf = MLPClassifier(random_state=43)
# fill in code below
blending_mlp_clf.fit(X_stack_train, y_train)
cross_val_score(blending_mlp_clf, X_stack_train, y_train, cv=3, scoring="accuracy")

c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\rache\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.96006749, 0.95537221, 0.95555972])

The random forest blender is better than MLP blender.

We now use random forest blender on the test data. (10 points)

In [74]:
# fill in code below
X_test_scaled = scaler.transform(X_test.astype(np.float64))

y_test_rnd_proba = rnd_clf.predict_proba(X_test_scaled)
y_test_mlp_proba = mlp_clf.predict_proba(X_test_scaled)
y_test_ovr_proba = ovr_clf.predict_proba(X_test_scaled)

# 2. Stack probability features horizontally (yields 78 columns)
X_stack_test_proba = np.c_[y_test_rnd_proba, y_test_mlp_proba, y_test_ovr_proba]

# 3. Predict and score using the probability-trained Random Forest blender
y_test_blending = blending_rnd_clf.predict(X_stack_test_proba)
print(blending_rnd_clf.__class__.__name__, accuracy_score(y_test, y_test_blending))

RandomForestClassifier 0.9735


This is the best classification result so far. We beat sklearn!

It took 28 minutes to run all the code.